In [91]:
%matplotlib widget
import pydicom as dicom
import glob
import os
import numpy as np

paths = sorted(glob.glob('data/4D-Lung/manifest-1758097641170/4D-Lung/101_HM10395/10-31-1997-NA-p4-67256/*'))
paths
npys = np.stack([dicom.dcmread(os.path.join(p, '1-25.dcm')).pixel_array for p in paths if os.path.isfile(os.path.join(p, '1-33.dcm'))])
# test = dicom.dcmread('data/4D-Lung/manifest-1758097641170/4D-Lung/101_HM10395/10-21-1997-NA-p4-86157/1.000000-P4P101S300I00003 Gated 0.0A-78180/1-001.dcm')

In [67]:
import os
import numpy as np
import pydicom
import re

def load_4d_ct(root_dir):
    """
    Load a 4D CT dataset where subfolders are named like:
    '501.000000-P4P101S100I0 Gated 10.0-77394'
    
    Returns:
        arr4d: NumPy array of shape (num_phases, num_slices, H, W) in Hounsfield Units.
        phases: Sorted list of phase labels (e.g. [10.0, 20.0, ...])
    """
    # Find subfolders that contain "Gated"
    gated_dirs = [d for d in os.listdir(root_dir) if "Gated" in d]

    # Extract phase number from folder names (e.g., "Gated 10.0" → 10.0)
    phase_pattern = re.compile(r"Gated\s+(\d+\.?\d*)")

    phase_info = []
    for d in gated_dirs:
        match = phase_pattern.search(d)
        if match:
            phase_num = float(match.group(1))
            phase_info.append((phase_num, os.path.join(root_dir, d)))

    # Sort by phase number
    phase_info.sort(key=lambda x: x[0])

    all_phases = []
    phases = []

    for phase_num, gated_path in phase_info:
        dcm_files = [os.path.join(gated_path, f) for f in os.listdir(gated_path) if f.endswith(".dcm")]

        # Read slices
        slices = [pydicom.dcmread(f) for f in dcm_files]

        # Sort slices by z-position
        slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))

        # Convert to Hounsfield Units (HU)
        slope = float(slices[0].RescaleSlope)
        intercept = float(slices[0].RescaleIntercept)

        vol = np.stack([s.pixel_array for s in slices]).astype(np.int16)
        vol_hu = vol * slope + intercept

        all_phases.append(vol_hu)
        phases.append(phase_num)

    # Final 4D array
    arr4d = np.stack(all_phases)

    return arr4d, phases


# Example usage
root_dir = "data/4D-Lung/manifest-1758097641170/4D-Lung/101_HM10395/10-31-1997-NA-p4-67256"
arr4d, phases = load_4d_ct(root_dir)

print("Final array shape:", arr4d.shape)  # (num_phases, num_slices, H, W)
print("Phases:", phases)
print("Example HU range:", arr4d.min(), "to", arr4d.max())


Final array shape: (10, 50, 512, 512)
Phases: [0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0]
Example HU range: -2669.0 to 2504.0


In [97]:
from src.vis_utils import create_animation_general
html, ani = create_animation_general(np.clip(arr4d[:, 25].astype(int), -1000, -500))